# Abfragen nach Kunden

## Zeige die unterschiedlichen Bezeichnungen für Positionen der Ansprechpartner in den Kunden-Firmen.

![Ku-0010](Screenshots/Ku-0010.png)

In [ ]:
--
SELECT DISTINCT POSITION
FROM kunde;

## Finde Kunden, deren Ansprechpartner eine Inhaber-, Leitungs- oder Manager-Position hat. (43 Zeilen)

![Ku-0020](Screenshots/Ku-0020.png)

In [ ]:
--
SELECT firma AS "Kunde", vorname + ' ' + nachname AS "Ansprechpartner", Position, Telefon
FROM KUNDE
WHERE Position LIKE '%Inhaber%'
   OR Position LIKE '%Leitungs%'
   OR Position LIKE '%Manager%';

## Zeige Firma, Ort und Land von Kunden, deren Firmenname **cocina** oder **comida** enthält.

![Ku-0030](Screenshots/Ku-0030.png)

In [ ]:
--
SELECT firma as 'Kunde', Ort, Land
FROM Kunde
WHERE firma LIKE '%cocina%' OR firma LIKE '%comida%';

## Zeige die Firmen der Kunden nach Ländern gruppiert an. (91 Zeilen)

Sortiere nach Land und Firmennamen.

![Ku-0040](Screenshots/Ku-0040.png)

In [ ]:
--
SELECT Land, Firma
FROM Kunde
ORDER BY Land ASC, Firma ASC;

## Zähle die Anzahl Kunden je Land (21 Zeilen)

Sortiere nach Land.

![Ku-0050](Screenshots/Ku-0050.png)

In [ ]:
--
SELECT Land, COUNT(Land) AS "Anzahl Kunden"
FROM Kunde
GROUP BY Land;

## Ranke die Länder nach Anzahl Kunden absteigend.

In der Nummerierung der Ränge sollen keine Lücken auftreten.

![Ku-0060](Screenshots/Ku-0060.png)

In [ ]:
--
SELECT
    DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) AS 'Rang',
    Land,
    COUNT(*) AS 'Anzahl Kunden'
FROM Kunde
GROUP BY Land
ORDER BY COUNT(*) DESC;

## Finde Kunden, die 2013 keine Bestellungen gemacht haben.

Zeige mehrere Lösungswege!

![Ku-0070](Screenshots/Ku-0070.png)

In [ ]:
-- unabhängige Unterabfrage
SELECT k.kunde_id, k.firma
FROM Kunde AS k
WHERE k.kunde_id NOT IN (
    SELECT b.kunde_id
    FROM Bestellung AS b
    WHERE YEAR(b.bestelldatum) = 2013
);

In [ ]:
-- korrelierte Unterabfrage
SELECT k.kunde_id, k.firma
FROM Kunde AS k
WHERE NOT EXISTS (
    SELECT 1
    FROM Bestellung AS b
    WHERE b.kunde_id = k.kunde_id
      AND YEAR(b.bestelldatum) = 2013
);

In [ ]:
-- JOIN
SELECT k.kunde_id, k.firma
FROM Kunde AS k
LEFT JOIN Bestellung AS b
    ON k.kunde_id = b.kunde_id
    AND YEAR(b.bestelldatum) = 2013
WHERE b.kunde_id IS NULL;

## 8. Zeige die Firmennamen **aller** Kunden mit der Anzahl ihrer Bestellungen in 2013 an.

Sortiere nach Anzahl absteigend. Finde alternative Lösungen. **(91 Zeilen)**

![Ku-0080](.\Screenshots\Ku-0080.png)

In [70]:
-- mit JOIN
SELECT k.firma, COUNT(b.bestellung_id) AS Anzahl_Bestellungen_2013
FROM Kunde AS k
LEFT JOIN Bestellung AS b
    ON k.kunde_id = b.kunde_id
    AND YEAR(b.bestelldatum) = 2013
GROUP BY k.firma
ORDER BY Anzahl_Bestellungen_2013 DESC;

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(91 rows affected)

firma                                | Anzahl_Bestellungen_2013
-------------------------------------+-------------------------
Let's Stop N Shop                    | 10                      
LINO-Delicateses                     | 8                       
Magazzini Alimentari Riuniti         | 8                       
Mère Paillarde                       | 8                       
North/South                          | 7                       
Old World Delicatessen               | 7                       
Paris spécialités                    | 7                       
Piccolo und mehr                     | 7                       
Lazy K Kountry Store                 | 7                       
Königlich Essen                      | 7                       
La maison d'Asie                     | 7                       
Drachenblut Delikatessen             | 7                     

In [ ]:
-- mit korrelierter Unterabfrage


## 9. Zeige für 2013 Kunden, Anzahl Bestellungen und Umsatz an.

Sortiere nach Firmenname. Formatiere den Umsatz als Currency. **(91 Zeilen)**

![Ku-0090](Screenshots/Ku-0090.png)

In [ ]:
--
SELECT
    k.firma,
    COUNT(DISTINCT b.bestellung_id) AS [Anzahl Bestellungen 2013],
    FORMAT(SUM(bd.verkaufspreis * bd.anzahl), 'C', 'de-DE') AS Umsatz_2013
FROM Kunde AS k
LEFT JOIN Bestellung AS b
    ON k.kunde_id = b.kunde_id
    AND YEAR(b.bestelldatum) = 2013
LEFT JOIN bestelldetail AS bd
    ON b.bestellung_id = bd.bestellung_id
GROUP BY k.kunde_id, k.firma
ORDER BY k.firma

## 10. Filtere die vorangehende Abfrage nach Kunden mit über 15 Tsd. Umsatz in 2013.

![Ku-0100](Screenshots/Ku-0100.png)

In [69]:
--
SELECT
    k.firma,
    COUNT(DISTINCT b.bestellung_id) AS [Anzahl Bestellungen 2013],
    FORMAT(SUM(bd.verkaufspreis * bd.anzahl), 'C', 'de-DE') AS Umsatz_2013
FROM Kunde AS k
LEFT JOIN Bestellung AS b
    ON k.kunde_id = b.kunde_id
    AND YEAR(b.bestelldatum) = 2013
LEFT JOIN bestelldetail AS bd
    ON b.bestellung_id = bd.bestellung_id
GROUP BY k.kunde_id, k.firma
HAVING SUM(bd.verkaufspreis * bd.anzahl) > 15000
ORDER BY k.firma

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(6 rows affected)

firma                        | Anzahl Bestellungen 2013 | Umsatz_2013
-----------------------------+--------------------------+------------
Centro comercial Moctezuma   | 6                        | 15.322,75 €
Familia Arquibaldo           | 6                        | 20.100,20 €
GROSELLA-Restaurante         | 7                        | 25.661,20 €
Hungry Owl All-Night Grocers | 6                        | 28.294,75 €
Let's Stop N Shop            | 10                       | 21.182,68 €
Piccolo und mehr             | 7                        | 19.769,50 €
(6 rows)

Total execution time: 00:00:00.034

## 11. Bewerte Kunden nach Ihrem Umsatz in 2013 wie folgt:

- ab 20000 : "Top"
- ab 15000 : "Super"
- ab 10000 : "Gut"
- ab 5000  : "OK"
- sonst : "Meh!"

**(91 Zeilen)**

![Ku-0101](.\Screenshots\Ku-0101.png)

In [ ]:
--


## 12. Ranke die Herkunftsländer der Kunden nach ihrem Umsatz in 2013 (21 Zeilen)

![Ku-0110](Screenshots/Ku-0110.png)

In [83]:
--
SELECT
    RANK() OVER (ORDER BY SUM(bd.verkaufspreis * bd.anzahl) DESC) AS Rang,
    k.land,
    SUM(bd.verkaufspreis * bd.anzahl) AS [Umsatz 2013]
FROM Kunde AS k
LEFT JOIN Bestellung AS b
    ON k.kunde_id = b.kunde_id
    AND YEAR(b.bestelldatum) = 2013
LEFT JOIN bestelldetail AS bd
    ON b.bestellung_id = bd.bestellung_id
GROUP BY k.land
ORDER BY Rang

Warnung: Ein NULL-Wert wird durch einen Aggregat- oder sonstigen SET-Vorgang gelöscht.
(21 rows affected)

Rang | land           | Umsatz 2013
-----+----------------+------------
1    | USA            | 93702,56   
2    | Brasilien      | 89123,49   
3    | Frankreich     | 69712,46   
4    | Deutschland    | 65726,67   
5    | Venezuela      | 62741,92   
6    | Großbritannien | 47493,73   
7    | Mexiko         | 36527,13   
8    | Irland         | 28294,75   
9    | Spanien        | 27810,70   
10   | Österreich     | 26531,60   
11   | Italien        | 20912,06   
12   | Argentinien    | 17882,65   
13   | Belgien        | 17122,05   
14   | Kanada         | 16188,25   
15   | Portugal       | 12074,04   
16   | Dänemark       | 10430,15   
17   | Schweden       | 6775,74    
18   | Schweiz        | 5976,55    
19   | Norwegen       | 3044,50    
20   | Finnland       | 317,75     
21   | Polen          | NULL       
(21 rows)

Total execution time: 00:00:00.016

## Pivotiere die Gesamtumsätze nach Ländern und Jahren (21 Zeilen)

![Ku-0120](Screenshots/Ku-0120.png)

In [ ]:
--


## Zeige für jedes Land die beiden Kunden mit dem höchsten Umsatz 2013 an. (39 oder 42 Zeilen)

![Ku-0130](Screenshots/Ku-0130.png)

In [ ]:
-- Lösung mit Tabellenausdruck in FROM


In [ ]:
-- Lösung mit Tabellenausdruck in WITH


In [ ]:
-- Lösung mit APPLY
